# 🧭 DT-HRES-S — 4D Methodology Walkthrough

This notebook walks through the **4D Methodology** stage by stage, showing the code that implements each dimension. It serves as both:

- **A tutorial** for new team members joining the project
- **A demo** for stakeholders (Dr. Rasikh, EPICS reviewers) to see the methodology applied

## What you'll learn

1. The conceptual framework (🥚 1D-2D) — what we're modeling and why
2. The data layer (💪 3D) — how the twin consumes data abstractly
3. The simulation (HRES 4) — physics-based digital shadow
4. The ML layer (HRES 7) — fast inference for interactive UI
5. The self-correction (HRES 6) — how the twin stays accurate over time
6. The Universe principles 🪐 — modularity, telemetry, uncertainty

---
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Aaron-Cuevas/DT-HRES-S/blob/main/notebooks/13_4D_methodology_walkthrough.ipynb)

In [ ]:
import os, sys
if 'google.colab' in sys.modules and not os.path.exists('DT-HRES-S'):
    !git clone -q https://github.com/Aaron-Cuevas/DT-HRES-S.git
    %cd DT-HRES-S
    !pip install -q -r requirements.txt
elif os.path.basename(os.getcwd()) == 'notebooks':
    sys.path.insert(0, '..')

from datetime import datetime, timedelta
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (12, 4)
plt.rcParams['axes.grid'] = True
print('✓ Setup complete')

## 🥚 1D — The Concept: physics equations made code

The first dimension is conceptual: what equations govern the system? The DT-HRES-S implements these as transparent Python functions you can read and understand.

In [ ]:
# Example: PV power equation
from src.pv_model import pv_dc_power, PVSystem

system = PVSystem(p_rated_W=400, n_panels=10, tilt_deg=20, gamma_pmpp=-0.004, noct_C=45)

# Try different conditions and see how PV power changes
conditions = [
    ('Cloudy day',        300, 25),
    ('Sunny morning',     600, 28),
    ('Sunny noon, cool',  950, 25),
    ('Sunny noon, HOT!',  950, 42),
]
print(f'{"Condition":25s} | POA (W/m²) | T_cell (°C) | P_pv (W)')
print('-' * 70)
for label, poa, t_cell in conditions:
    p = float(pv_dc_power(np.array([float(poa)]), np.array([float(t_cell)]), system)[0])
    print(f'{label:25s} | {poa:>10} | {t_cell:>11} | {p:>8.0f}')

Notice: at the same POA (950 W/m²), going from 25°C to 42°C drops PV output by ~7%. That's the temperature coefficient (γ ≈ -0.4%/°C) at work — the equation is doing exactly what the physics dictates.

## 💪 3D — The Mind: data sources abstracted away

The twin doesn't care WHERE its weather data comes from. Let's see this in action.

In [ ]:
from src.sensors import TMYFileSource, NASAPowerSource, PiranometerSource

# Tier 1: synthetic TMY (works today)
tmy = TMYFileSource('data/processed/campeche_tmy.csv', 'Campeche')
print('=== Tier 1: TMY File ===')
print('Metadata:', tmy.metadata())
print()
reading = tmy.read(datetime(2026, 6, 15, 13, 0))
print('Reading:', reading)
print()

# Tier 2: NASA POWER (skeleton, pending Samuel's implementation)
nasa = NASAPowerSource(lat=21.12, lon=-88.73)  # Ixil coordinates
print('=== Tier 2: NASA POWER ===')
print('Metadata:', nasa.metadata())
print('(read() not yet implemented — pending Task 3.2)')
print()

# Tier 3: physical sensor (HRES 5, pending hardware)
sensor = PiranometerSource(device_id='IXIL_PYRANO_01')
print('=== Tier 3: Physical pyranometer ===')
print('Metadata:', sensor.metadata())
print('(read() not yet implemented — pending field deployment)')

**Key insight:** All three sources share the same interface (`read()` + `metadata()`). The simulator's code doesn't change when we swap them — that's the **Modularity** principle of the Universe.

## 🧠 HRES 4 — Digital Shadow in action

Run the full HRES simulation: PV + Wind + Battery + community demand.

In [ ]:
from src.data_loader import load_city
from src import pv_model, wind_model, battery_model, hres_simulator as hres

df = load_city('Campeche')  # proxy for Ixil
config = hres.HRESConfig(
    pv=pv_model.PVSystem(p_rated_W=400, n_panels=10, tilt_deg=20),
    wind=wind_model.WindTurbine(rated_power_W=3000),
    battery=battery_model.Battery(capacity_kWh=15),
)
sim = hres.run(df, config)
kpis = hres.summarize(sim, config)

print('Annual KPIs for a 4 kWp PV + 3 kW wind + 15 kWh battery system in Campeche/Ixil:')
for k, v in kpis.items():
    fmt = f'{v:.2f}' if isinstance(v, float) else str(v)
    print(f'  {k:30s}: {fmt}')

## 👻 HRES 7 — ML predicts (now-casting)

Train and benchmark the 4 ML models. The Random Forest typically wins.

In [ ]:
from src.ml_models import benchmark
results = benchmark(sim, target_col='p_pv_W')
print('Performance comparison (predicting hourly PV power):')
results.round(3)

## ❓ Uncertainty — how confident is the twin?

Every prediction comes with an uncertainty budget.

In [ ]:
from src.uncertainty import full_budget_for_pv, full_budget_for_wind

# PV prediction with TMY input quality
b_pv = full_budget_for_pv('TMY_file', model_uncertainty_pct=5.4, aleatoric_pct=2.0)
print('=== PV uncertainty budget ===')
print(f'  Input (sensor):    ±{b_pv.input_pct:.1f}%')
print(f'  Model (ML):        ±{b_pv.model_pct:.1f}%')
print(f'  Aleatoric (noise): ±{b_pv.aleatoric_pct:.1f}%')
print(f'  COMBINED:          ±{b_pv.combined_pct:.1f}%')
print(f'  Example: {b_pv.summary(3000)}')
print()

# Wind is much worse because P ∝ v³
b_w = full_budget_for_wind('TMY_file')
print('=== Wind uncertainty budget ===')
print(f'  COMBINED:          ±{b_w.combined_pct:.1f}%')
print(f'  Example: {b_w.summary(1800)}')
print()
print('Wind uncertainty is much larger because P_wind ∝ v³.')

## 👻 HRES 6 — Auto-correction (drift detection)

Once real measurements are available, the twin detects when it drifts and recommends retraining.

In [ ]:
from src.auto_correction import AutoCorrector

corrector = AutoCorrector(model_version='rf_v0.2.0',
                          retraining_threshold_pct=15.0,
                          minimum_samples=50)

# Simulate 100 hours of predictions and measurements
# Scenario: the model overestimates by 20% (drift!)
base = datetime(2026, 8, 1, 12, 0)
for i in range(120):
    ts = base + timedelta(hours=i)
    true_value = 2500 + 800 * np.sin(i / 24 * 2 * np.pi)  # daily cycle
    if true_value < 50:  # skip night
        continue
    predicted = true_value * 1.20
    corrector.log_prediction(ts, predicted)
    corrector.log_measurement(ts, true_value)

drift_report = corrector.check_drift()
print('=== Drift detection report ===')
for k, v in drift_report.items():
    print(f'  {k}: {v}')

The corrector detected the 20% overestimate and recommended retraining. In production, this would trigger an automated pipeline that re-fits the model with the accumulated paired data.

## 📡 Telemetry — every event is logged

Universe principle of **Tracking**: every prediction is stored with full provenance.

In [ ]:
from src.telemetry import TelemetryLogger, read_log

with TelemetryLogger(session_id='walkthrough_demo') as tlog:
    tlog.log_prediction(
        timestamp=datetime.now(),
        features={'ghi_Wm2': 850, 'dry_bulb_C': 32, 'wind_speed_ms': 5.5,
                  'hour': 13, 'month': 6},
        prediction=3247.5,
        model_version='rf_v0.2.0',
        uncertainty_pct=7.6,
        source_metadata=tmy.metadata(),
    )
    tlog.log_drift_alert(
        mean_error_pct=drift_report.get('mean_error_pct', 0),
        samples_evaluated=drift_report.get('samples_evaluated', 0),
        threshold_pct=15.0,
        action=drift_report.get('action', 'unknown'),
    )

# Read back the log
log_df = read_log(tlog.log_path)
print(f'Log entries: {len(log_df)}')
print(log_df[['timestamp_utc', 'event_type']].to_string(index=False))

## 📊 Summary

You just saw the 4D methodology applied:

| Stage | Demonstrated by |
|---|---|
| 🥚 1D — Concept | `pv_dc_power()` evaluating physics equations |
| 🥚 2D — Body | (see notebook 12 — community interface) |
| 💪 3D — Mind | 3-tier data source abstraction |
| 💪 4D — Spirit | (see `docs/4D_methodology/04_spirit_physical_arrangement.md`) |
| 🧠 HRES 4 — Digital Shadow | Full simulator runs with synthetic data |
| 🧠 HRES 5 — Real Data | (skeleton ready, pending field campaign) |
| 👻 HRES 6 — Auto-correction | Drift detection identified the 20% bias |
| 👻 HRES 7 — ML Forecasting | 4 models benchmarked, Random Forest wins |
| 🪐 Modularity | Code in 10 independent modules |
| 🪐 Telemetry | Every event logged with provenance |
| 🪐 Tracking | Sensor metadata propagated |
| 🪐 APIs | (skeleton in `src/api.py` — v0.3) |
| 🪐 Uncertainty | Combined error budget delivered with every prediction |

---

**Next:** see `notebooks/12_community_interface.ipynb` for the interactive community-facing UI.

📚 Full documentation: [`docs/4D_methodology/`](../docs/4D_methodology/)